# DPPUv7 paper04: Paper Figures
## Euclidean Geometric Response Dictionary across 4 Thurston Geometries

This notebook generates five figures for paper04 using the DPPU library
(`dppu/`) and standalone numerical routines that mirror the
verification scripts in `scripts/paper04/` and `scripts/proofs/`.

### Figure map

| Figure | Section | Content | Engine input |
|--------|---------|---------|--------------|
| Fig. 2 | §3.3 / R4 | EC slice potential $V_{\rm eff}\|_{\eta=V=0}$ across 4 geometries | `dppu.action.ec_action.build_veff_ec` |
| Fig. 3 | §3.4 / R5 | Reduced 1D $\eta$-mode defect localization (4 geometries × 3 dip profiles) | `dppu` engine + Sturm-Liouville eigensolver |
| Fig. 4 | §4.2 / B2 | $Nil^3$ Heisenberg Landau spectrum and $\eta_{\rm APS}^{(3D)}=+1/2$ construction | analytic ladder + Levi-Civita spinor CS integral |
| Fig. 5 | §4.2 / B4 | $Sol^3$ global spectral branch on the compact mapping torus $M_A=T^2\rtimes_A\!S^1$ | local CS + hyperbolic monodromy + spin-structure kernel |
| Fig. 7 | §4.5 | scaffold-vs-CS scatter | table-derived scaffold data |

The numbering follows the figure plan in paper04. Fig. 1, 6, and 8 are
schematic / table-derived Mermaid diagrams produced in the Markdown draft.

**Author:** Muacca
**Date:** 2026-04-25

---

> **Computation time:**
> - `[Engine]` cell (first run): ~30–60 s (dominated by the SymPy `dppu` engines).
> - Subsequent runs: instant from `data/paper04_figures_cache.pkl`.
> - Figure cells: instant.


In [ ]:
import sys
import os
import pickle
import time
import math
import ast

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh
from scipy.linalg import expm

# ── Matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 110,
    "savefig.dpi": 200,
})

# ── Path setup ────────────────────────────────────────────────
# Notebook lives at script/scripts/visualize/  -> project root is ../..
notebook_dir = os.path.abspath("")
project_root = os.path.normpath(os.path.join(notebook_dir, "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

output_dir = os.path.normpath(os.path.join(project_root, "../LaTeX/figures"))
cache_dir = os.path.normpath(os.path.join(project_root, "../data"))
cache_file = os.path.join(cache_dir, "paper04_figures_cache.pkl")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Output dir   : {output_dir}")
print(f"Cache dir    : {cache_dir}")

# ── Cache helpers ─────────────────────────────────────────────
def load_cache():
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    return {}

def save_cache(c):
    with open(cache_file, "wb") as f:
        pickle.dump(c, f)

cache = load_cache()
print(f"Cached keys  : {list(cache.keys())}")


## [Engine] DPPU + numerical computations (cached)

The engine cell prepares the data for the engine-derived figures.

| Block | Cost | What it produces |
|-------|------|------------------|
| Fig.2 | medium (4 × `build_veff_ec`) | EC slice potential $V_{\rm slice}(R; \alpha=-1)$ for 4 geometries on $\eta=V=0$; analytic stationary radius $R_0=4/\sqrt{3}$ for $Nil^3,Sol^3$. |
| Fig.3 | medium (4 × `dppu` engine + 12 × SL eigensolver) | $c_{\rm geo}$ for $T^3, Nil^3, S^3, Sol^3$; lowest eigenvalue $E_{\min}$ for 4 × 3 = 12 cases; ground-state wavefunction for the representative case $A=0.5,\,w=2\rho_0$. |
| Fig.4 | low (NumPy linear algebra) | $Nil^3$ Heisenberg Landau spectrum $\mu_n^{\uparrow,\downarrow}(p_2)$; explicit Levi-Civita spinor CS integral $\int_Y \mathcal{C}_3^{\sigma}$; assembled $\eta_{\rm APS}=+1/2$, $h=0$, $\eta(0)=1$. |
| Fig.5 | low (matrix exp + bookkeeping) | $Sol^3$ local CS = 0; fiber mass profile $m_k(t)=\|p_k(t)\|$ for representative $k\neq0$; kernel dimensions $h(\text{Sol-P})=2$, $h(\text{Sol-A})=0$; assembled $\eta_{\rm APS}=1/0$. |
| Fig.7 | none | table-derived scaffold-vs-CS scatter; generated without engine input. |

All computations mirror the reproducibility scripts in `scripts/paper04/` and `scripts/proofs/`. No physical values are hardcoded — they are computed from the DPPU engine or from explicit ladder / linear-algebra routines.


In [ ]:
# ── [Engine] cached computations for Figs. 2, 3, 4, 5 ─────────────────────

NEED_FIG2 = "fig2_ec_slice" not in cache
NEED_FIG3 = "fig3_defect"   not in cache
NEED_FIG4 = "fig4_nil3_aps" not in cache
NEED_FIG5 = "fig5_sol3_aps" not in cache

if NEED_FIG2 or NEED_FIG3 or NEED_FIG4 or NEED_FIG5:
    print("[ENGINE] Running computations...")
    t0_total = time.time()

    from sympy import S, cancel, diff, lambdify, simplify

    from dppu.topology.unified import DOFConfig, TopologyType, UnifiedEngine
    from dppu.torsion.mode import Mode
    from dppu.torsion.nieh_yan import NyVariant
    from dppu.action.ec_action import build_veff_ec

    TOPOLOGIES = [
        ("T^3",   TopologyType.T3),
        ("Nil^3", TopologyType.NIL3),
        ("S^3",   TopologyType.S3),
        ("Sol^3", TopologyType.SOL3),
    ]

    # ─────────────────────────────────────────────────────────────────────
    # Fig. 3 — reduced 1D eta-mode defect localization
    # ─────────────────────────────────────────────────────────────────────
    if NEED_FIG3:
        print("  [Fig.3] eta-mode SL eigenproblem...", end="", flush=True)
        t1 = time.time()

        # c_geo from the exact homogeneous mass datum (kappa = L = 1)
        c_geo_dict = {}
        for label, topo in TOPOLOGIES:
            cfg = DOFConfig(
                topology=topo,
                torsion_mode=Mode.AX,
                enable_squash=False,
                enable_shear=False,
                ny_variant=NyVariant.FULL,
            )
            eng = UnifiedEngine(cfg)
            eng.run()
            params = eng.data["params"]
            rho = params.get("R", params["r"])
            veff = eng.data["potential"].subs({
                params["V"]: S.Zero,
                params["theta_NY"]: S.Zero,
            })
            m_geo = simplify(diff(veff, params["eta"], 2).subs(params["eta"], 0))
            c_sym = simplify(m_geo / rho).subs({
                params["L"]: S.One,
                params["kappa"]: S.One,
            })
            c_geo_dict[label] = float(c_sym)

        # Three Gaussian dip profiles
        profiles = [
            ("A=0.3, w=1.0 r0", 0.3, 1.0),
            ("A=0.5, w=2.0 r0", 0.5, 2.0),
            ("A=0.7, w=1.0 r0", 0.7, 1.0),
        ]

        rho0 = 3.0
        z_max = 6.0 * rho0
        N_grid = 2001
        z = np.linspace(-z_max, z_max, N_grid)
        dz = z[1] - z[0]

        def rho_dip(z, A, w):
            return rho0 * (1.0 - A * np.exp(-(z / w) ** 2))

        def sl_solve(rho_arr, c_geo, dz, n_eigs=1):
            K = c_geo * rho_arr
            M = c_geo * rho_arr
            Kh = 0.5 * (K[:-1] + K[1:])
            N = len(rho_arr)
            diag = np.zeros(N)
            diag[1:-1] = (Kh[:-1] + Kh[1:]) / dz**2 + M[1:-1]
            diag[0]    = Kh[0]  / dz**2 + M[0]
            diag[-1]   = Kh[-1] / dz**2 + M[-1]
            off = -Kh / dz**2
            H = diags([off, diag, off], offsets=[-1, 0, 1], format="csr")
            vals, vecs = eigsh(H, k=n_eigs, which="SA")
            order = np.argsort(vals)
            return vals[order], vecs[:, order]

        results = {}
        for glabel, _ in TOPOLOGIES:
            c_geo = c_geo_dict[glabel]
            for plabel, A_amp, wr in profiles:
                w = wr * rho0
                rho_arr = rho_dip(z, A_amp, w)
                vals, _ = sl_solve(rho_arr, c_geo, dz, n_eigs=1)
                M0 = c_geo * rho0
                results[(glabel, plabel)] = (float(vals[0]), float(M0))

        # Representative wavefunction (geometry-invariant shape in normalized units)
        rep_label = "A=0.5, w=2.0 r0"
        A_rep, wr_rep = 0.5, 2.0
        w_rep = wr_rep * rho0
        rho_rep = rho_dip(z, A_rep, w_rep)
        c_norm = c_geo_dict["Nil^3"]
        vals_rep, vecs_rep = sl_solve(rho_rep, c_norm, dz, n_eigs=1)
        psi0 = vecs_rep[:, 0]
        psi0 = psi0 / np.sqrt(np.trapezoid(psi0**2, z))
        if psi0[len(psi0) // 2] < 0:
            psi0 = -psi0

        cache["fig3_defect"] = dict(
            z=z.tolist(),
            rho_profiles={
                p[0]: rho_dip(z, p[1], p[2] * rho0).tolist() for p in profiles
            },
            c_geo=c_geo_dict,
            results={f"{g}|{p}": v for (g, p), v in results.items()},
            psi0_rep=psi0.tolist(),
            rho_rep=rho_rep.tolist(),
            E0_rep=float(vals_rep[0]),
            M0_rep=float(c_norm * rho0),
            rho0=rho0,
            profile_labels=[p[0] for p in profiles],
            geom_labels=[g for g, _ in TOPOLOGIES],
            rep_label=rep_label,
        )
        save_cache(cache)
        print(f" done ({time.time()-t1:.1f}s)")

    # ─────────────────────────────────────────────────────────────────────
    # Fig. 4 — Nil^3 Heisenberg Landau spectrum and APS construction
    # ─────────────────────────────────────────────────────────────────────
    if NEED_FIG4:
        print("  [Fig.4] Nil^3 Landau levels + spinor CS...", end="", flush=True)
        t1 = time.time()

        r0_nil = 3.0

        def heisenberg_levels(p2, r0, n_max=4):
            k2 = 2.0 * math.pi * p2 / r0
            k2a = abs(k2)
            omega = k2a / r0
            up   = np.array([2.0 *  n      * omega * k2a + k2a**2 for n in range(n_max + 1)])
            down = np.array([2.0 * (n + 1) * omega * k2a + k2a**2 for n in range(n_max + 1)])
            return up, down

        ppa_modes = [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
        n_max_show = 3
        spectrum = {}
        for p2 in ppa_modes:
            up, down = heisenberg_levels(p2, r0_nil, n_max=n_max_show)
            spectrum[str(p2)] = dict(up=up.tolist(), down=down.tolist())

        # Levi-Civita spinor CS integral on Nil^3 (mirrors eta_aps_nil3.py)
        def eps3(i, j, k):
            if (i, j, k) in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
                return 1
            if (i, j, k) in [(0, 2, 1), (2, 1, 0), (1, 0, 2)]:
                return -1
            return 0

        C = np.zeros((3, 3, 3))
        C[2, 0, 1] =  1.0 / r0_nil
        C[2, 1, 0] = -1.0 / r0_nil

        omega_c = np.zeros((3, 3, 3))
        for a in range(3):
            for b in range(3):
                for cc in range(3):
                    omega_c[a, b, cc] = 0.5 * (
                        C[a, b, cc] + C[cc, b, a] - C[b, a, cc]
                    )

        gamma = [
            np.array([[0, 1], [1, 0]],   dtype=complex),
            np.array([[0, -1j], [1j, 0]], dtype=complex),
            np.array([[1, 0], [0, -1]],  dtype=complex),
        ]
        sigma_gen = {}
        for a in range(3):
            for b in range(a + 1, 3):
                comm = gamma[a] @ gamma[b] - gamma[b] @ gamma[a]
                sigma_gen[(a, b)] =  0.5 * comm
                sigma_gen[(b, a)] = -sigma_gen[(a, b)]

        A_sigma = [np.zeros((2, 2), dtype=complex) for _ in range(3)]
        for cc in range(3):
            for a in range(3):
                for b in range(a + 1, 3):
                    A_sigma[cc] += omega_c[a, b, cc] * sigma_gen[(a, b)]

        dA_sigma = [[np.zeros((2, 2), dtype=complex) for _ in range(3)] for _ in range(3)]
        for d in range(3):
            for e in range(3):
                for cc in range(3):
                    dA_sigma[d][e] += A_sigma[cc] * (-C[cc, d, e])

        cs_A = 0.0
        for cc in range(3):
            for d in range(3):
                for e in range(d + 1, 3):
                    ep = eps3(cc, d, e)
                    if ep == 0:
                        continue
                    cs_A += ep * float(np.real(np.trace(A_sigma[cc] @ dA_sigma[d][e])))

        cs_B = 0.0
        for c1 in range(3):
            for c2 in range(3):
                for c3 in range(3):
                    ep = eps3(c1, c2, c3)
                    if ep == 0:
                        continue
                    cs_B += (2.0 / 3.0) * ep * float(
                        np.real(np.trace(A_sigma[c1] @ A_sigma[c2] @ A_sigma[c3]))
                    )

        vol = r0_nil**3
        cs_sigma_total = (cs_A + cs_B) * vol
        eta_APS = -cs_sigma_total
        h_nil = 0
        eta0_nil = 2.0 * eta_APS - h_nil

        cache["fig4_nil3_aps"] = dict(
            r0=r0_nil,
            ppa_modes=ppa_modes,
            n_max_show=n_max_show,
            spectrum=spectrum,
            cs_A=float(cs_A * vol),
            cs_B=float(cs_B * vol),
            cs_sigma_total=float(cs_sigma_total),
            eta_APS=float(eta_APS),
            h=int(h_nil),
            eta0=float(eta0_nil),
        )
        save_cache(cache)
        print(f" done ({time.time()-t1:.1f}s)")

    # ─────────────────────────────────────────────────────────────────────
    # Fig. 5 — Sol^3 compact mapping-torus benchmark
    # ─────────────────────────────────────────────────────────────────────
    if NEED_FIG5:
        print("  [Fig.5] Sol^3 mapping torus...", end="", flush=True)
        t1 = time.time()

        # The local CS_sigma vanishes on Sol^3 (verified by proofs/eta_aps_sol3.py).
        # We re-record the value here as a flag.
        cs_sigma_local = 0.0

        # Hyperbolic monodromy A = [[2,1],[1,1]]
        A_mat = np.array([[2.0, 1.0], [1.0, 1.0]])
        vals_A, vecs_A = np.linalg.eig(A_mat)
        L_log = np.real_if_close(
            vecs_A @ np.diag(np.log(vals_A)) @ np.linalg.inv(vecs_A)
        )

        t_grid = np.linspace(0.0, 1.0, 200)
        sample_modes = [(1, 0), (0, 1), (1, 1), (2, 1)]
        m_profiles = {}
        for k in sample_modes:
            prof = np.array([
                np.linalg.norm(expm(t * L_log.T) @ np.array(k, dtype=float))
                for t in t_grid
            ])
            m_profiles[str(k)] = prof.tolist()

        h_solp = 2  # base periodic   (Sol-P)
        h_sola = 0  # base anti-periodic (Sol-A)
        # eta(0) = 0 for both via time-reversal antiunitary symmetry
        eta_solp = (0 + h_solp) / 2
        eta_sola = (0 + h_sola) / 2

        cache["fig5_sol3_aps"] = dict(
            cs_sigma_local=float(cs_sigma_local),
            t_grid=t_grid.tolist(),
            m_profiles=m_profiles,
            sample_modes=[list(k) for k in sample_modes],
            h_solp=int(h_solp),
            h_sola=int(h_sola),
            eta_solp=float(eta_solp),
            eta_sola=float(eta_sola),
        )
        save_cache(cache)
        print(f" done ({time.time()-t1:.1f}s)")

    # ─────────────────────────────────────────────────────────────────────
    # Fig. 2 — EC slice potential V_slice(R) for 4 geometries
    # ─────────────────────────────────────────────────────────────────────
    if NEED_FIG2:
        print("  [Fig.2] EC slice potential...", end="", flush=True)
        t1 = time.time()

        rho_arr = np.linspace(0.5, 8.0, 400)
        curves = {}
        slice_repr = {}

        for label, topo in TOPOLOGIES:
            v, _, params = build_veff_ec(
                topo, torsion_mode=Mode.MX, ny_variant=NyVariant.FULL
            )
            rho = params.get("R", params["r"])
            slice_v = cancel(v.subs({
                params["eta"]: 0,
                params["V"]: 0,
                params["theta_NY"]: 0,
            }))
            slice_repr[label] = str(slice_v)
            slice_evald = slice_v.subs({
                params["kappa"]: 1,
                params["L"]: 1,
                params["alpha"]: -1,
            })
            f = lambdify(rho, slice_evald, modules="numpy")
            try:
                y = np.asarray(f(rho_arr), dtype=float)
                if y.ndim == 0:
                    y = np.full_like(rho_arr, float(y))
            except TypeError:
                # Constant expression (T^3: V_slice = 0)
                y = np.full_like(rho_arr, float(slice_evald))
            curves[label] = y.tolist()

        # Stationary radius for Nil^3 / Sol^3 at alpha=-1, kappa=1:
        # R_0 = 4/sqrt(3) sqrt(|alpha|)
        r0_branch = 4.0 / math.sqrt(3.0)

        cache["fig2_ec_slice"] = dict(
            rho_arr=rho_arr.tolist(),
            curves=curves,
            slice_repr=slice_repr,
            r0_branch=float(r0_branch),
        )
        save_cache(cache)
        print(f" done ({time.time()-t1:.1f}s)")

    print(f"[ENGINE] Done. Total: {time.time()-t0_total:.1f}s.")

else:
    print("All data loaded from cache.")

# Unpack
fig3 = cache["fig3_defect"]
fig4 = cache["fig4_nil3_aps"]
fig5 = cache["fig5_sol3_aps"]
fig2 = cache["fig2_ec_slice"]

print()
print("Summary:")
print(f"  Fig.3 c_geo (kappa=L=1):")
for k, v in fig3["c_geo"].items():
    print(f"    {k:6s} = {v:12.4f}")
print(f"  Fig.4 eta_APS(Nil^3, PPA) = {fig4['eta_APS']:+.6f}   "
      f"(eta(0) = {fig4['eta0']:+.0f}, h = {fig4['h']})")
print(f"  Fig.5 Sol-P: h = {fig5['h_solp']}, eta_APS = {fig5['eta_solp']:.0f}")
print(f"        Sol-A: h = {fig5['h_sola']}, eta_APS = {fig5['eta_sola']:.0f}")
print(f"  Fig.2 V_slice (alpha=-1, kappa=L=1):")
for k, v in fig2["slice_repr"].items():
    print(f"    {k:6s} : {v}")


## Fig. 2 — §3.3 / R4: EC slice potential across 4 geometries

**Source:** `dppu.action.ec_action.build_veff_ec`
**Section:** §3.3 (R4)

The current DPPU EC+NY+Weyl potential, restricted to the homogeneous
$\eta = V = 0$ slice with $\theta_{\rm NY} = 0$, gives geometry-dependent
$V_{\rm slice}(R)$. At $\alpha=-1,\,\kappa=L=1$:

- **$T^3$:** flat zero slice, $V_{\rm slice} \equiv 0$ — no isolated radial branch.
- **$S^3$:** non-vanishing constant slope on the round slice — no stationary point.
- **$Nil^3$:** $V_{\rm slice} = 4\pi^4 R - 64\pi^4\alpha/(3R)$, with isolated
  EC slice minimum at $R_0 = (4/\sqrt{3})\sqrt{|\alpha|}$.
- **$Sol^3$:** $V_{\rm slice} = 4 \times Nil^3$, same $R_0$, deeper potential.

This visual confirms R4: the EC slice-minimum branch exists only on $Nil^3$
and $Sol^3$, supplying the spin-0 EC entry that distinguishes them from
$T^3$ (flat / inert) and round $S^3$ (no stationary point).


In [ ]:
# ── Fig. 2 — EC slice potential V_slice(R) across 4 geometries ───────────
plt.close("all")

rho_arr = np.array(fig2["rho_arr"])
curves = {k: np.array(v) for k, v in fig2["curves"].items()}
slice_repr = fig2["slice_repr"]
r0_branch = fig2["r0_branch"]

pi4 = np.pi**4

styles = {
    "T^3":   dict(color="#888888", lw=2.0,
                  label=r"$T^3$ : $V_{\rm slice} \equiv 0$  (flat)"),
    "Nil^3": dict(color="#2196F3", lw=2.4,
                  label=r"$Nil^3$ : $4\pi^4 R - \frac{64\pi^4\alpha}{3R}$"),
    "S^3":   dict(color="#F44336", lw=2.0,
                  label=r"$S^3$ : monotone slope (no stationary point)"),
    "Sol^3": dict(color="#9C27B0", lw=2.4,
                  label=r"$Sol^3$ : $4 \times Nil^3$"),
}

fig, ax = plt.subplots(figsize=(8.5, 5.6))

for label, st in styles.items():
    y = curves[label] / pi4
    ax.plot(rho_arr, y, **st)

# Mark stationary points for Nil^3 / Sol^3
i_branch = int(np.argmin(np.abs(rho_arr - r0_branch)))
y_min_nil = curves["Nil^3"][i_branch] / pi4
y_min_sol = curves["Sol^3"][i_branch] / pi4

ax.plot(r0_branch, y_min_nil, "o", color="#2196F3",
        ms=10, mec="white", mew=1.0, zorder=6)
ax.plot(r0_branch, y_min_sol, "o", color="#9C27B0",
        ms=10, mec="white", mew=1.0, zorder=6)
ax.axvline(r0_branch, color="gray", ls=":", lw=0.8, alpha=0.55)

ax.annotate(
    r"$R_0 = \frac{4}{\sqrt{3}}\sqrt{|\alpha|}$",
    xy=(r0_branch, y_min_sol),
    xytext=(r0_branch + 1.2, y_min_sol - 12),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
    fontsize=12,
)

ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel(r"Radial scale  $R$  ($\kappa = L = 1$)")
ax.set_ylabel(r"$V_{\rm slice}(R;\,\alpha=-1)\,/\,\pi^4$")
ax.set_title(
    r"EC slice potential $V_{\rm eff}|_{\eta=V=0}$ across 4 geometries"
)

ax.set_xlim(0.5, 8.0)
ymin = min(y.min() for y in curves.values()) / pi4
ymax = max(y.max() for y in curves.values()) / pi4
ax.set_ylim(ymin * 1.1 if ymin < 0 else -2, max(ymax * 1.05, 60))
ax.legend(loc="upper right", framealpha=0.95)
ax.grid(True, alpha=0.25)

plt.tight_layout()
out_path = os.path.join(output_dir, "fig02_ec_slice_potential.png")
plt.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

print()
print("Slice potential expressions (theta_NY = eta = V = 0):")
for k, v in slice_repr.items():
    print(f"  {k:6s} : V_slice = {v}")
print(f"\nStationary radius (Nil^3, Sol^3 at alpha=-1, kappa=1):"
      f"  R_0 = 4/sqrt(3) = {r0_branch:.6f}")


## Fig. 3 — §3.4 / R5: Reduced 1D $\eta$-mode defect localization

**Source:** `dppu` engine (exact $M_{\rm geo}$) + Sturm-Liouville eigensolver
**Section:** §3.4 (R5)

The benchmark operator on a slice-wise homogeneous background is

$$
-\frac{d}{dz}\!\left[K_{\rm geo}(z)\frac{df}{dz}\right] + M_{\rm geo}(z)\,f = E\,f,
\qquad K_{\rm geo}(z) = M_{\rm geo}(z) = c_{\rm geo}\,\rho(z),
$$

with topology-dependent $c_{\rm geo}$ from the engine ($\kappa=L=1$):

- $c_{T^3} = c_{Nil^3} = c_{Sol^3} = 96\pi^4 \approx 9351.0$
- $c_{S^3} = 12\pi^2 \approx 118.4$.

Three representative Gaussian $\rho$-dips
$\rho(z) = \rho_0\bigl(1 - A\,e^{-z^2/w^2}\bigr)$ are sampled.

**Panel layout:**

- (a) The three $\rho(z)/\rho_0$ profiles.
- (b) Lowest eigenvalue $E_{\min}/M_0$ for $4 \times 3 = 12$ cases.
  All are $<1$, confirming the variational localization claim from
  Appendix E.6 ("12 of 12 LOCALIZED").
- (c) Ground-state wavefunction for the representative case
  $A=0.5,\,w=2\rho_0$, overlaid on $\rho(z)/\rho_0$.


In [ ]:
# ── Fig. 3 — reduced 1D eta-mode defect localization ─────────────────────
plt.close("all")

z = np.array(fig3["z"])
rho_profiles = {k: np.array(v) for k, v in fig3["rho_profiles"].items()}
results = fig3["results"]
rho0 = fig3["rho0"]
psi0 = np.array(fig3["psi0_rep"])
rho_rep = np.array(fig3["rho_rep"])
profile_labels = fig3["profile_labels"]
geom_labels = fig3["geom_labels"]

geom_colors = {
    "T^3":   "#888888",
    "Nil^3": "#2196F3",
    "S^3":   "#F44336",
    "Sol^3": "#9C27B0",
}
profile_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, axes = plt.subplots(3, 1, figsize=(8.6, 11.5))

# ── (a) Gaussian dip profiles ────────────────────────────────────────────
ax = axes[0]
for plabel, color in zip(profile_labels, profile_colors):
    rho_arr = rho_profiles[plabel]
    ax.plot(z / rho0, rho_arr / rho0, lw=2.0, color=color, label=plabel)
ax.axhline(1.0, color="gray", lw=0.7, ls=":")
ax.set_xlabel(r"$z/\rho_0$")
ax.set_ylabel(r"$\rho(z)/\rho_0$")
ax.set_title(r"(a) Gaussian $\rho$-dip profiles")
ax.set_xlim(-6, 6)
ax.set_ylim(0.2, 1.05)
ax.legend(loc="lower right", framealpha=0.95)
ax.grid(True, alpha=0.25)

# ── (b) E_min / M_0 for all 12 cases ─────────────────────────────────────
ax = axes[1]
n_g = len(geom_labels)
n_p = len(profile_labels)
width = 0.20
x_centers = np.arange(n_p)

for i, glabel in enumerate(geom_labels):
    ratios = []
    for plabel in profile_labels:
        E, M0 = results[f"{glabel}|{plabel}"]
        ratios.append(E / M0)
    offset = (i - (n_g - 1) / 2) * width
    ax.bar(
        x_centers + offset, ratios, width=width,
        color=geom_colors[glabel], edgecolor="white",
        label=f"${glabel}$",
    )

ax.axhline(1.0, color="red", lw=1.4, ls="--",
           label=r"$M_0$ (continuum threshold)")
ax.set_xticks(x_centers)
ax.set_xticklabels(profile_labels, fontsize=9)
ax.set_ylabel(r"$E_{\min}\,/\,M_0$")
ax.set_ylim(0.0, 1.10)
ax.set_title(r"(b) Bound-state energy / continuum (12/12 LOCALIZED)")
ax.legend(fontsize=9, loc="upper right", ncol=2, framealpha=0.95)
ax.grid(True, alpha=0.25, axis="y")

# ── (c) Ground-state wavefunction (geometry-invariant shape) ─────────────
ax = axes[2]
ax2 = ax.twinx()
line_psi, = ax.plot(
    z / rho0, psi0, color="#E91E63", lw=2.2,
    label=r"$\psi_0(z)$ (rep. case)",
)
line_rho, = ax2.plot(
    z / rho0, rho_rep / rho0, color="gray", lw=1.2, ls="--", alpha=0.7,
    label=r"$\rho(z)/\rho_0$",
)
ax.set_xlabel(r"$z/\rho_0$")
ax.set_ylabel(r"$\psi_0(z)$  (normalized)", color="#E91E63")
ax2.set_ylabel(r"$\rho(z)/\rho_0$", color="gray")
ax.tick_params(axis="y", labelcolor="#E91E63")
ax2.tick_params(axis="y", labelcolor="gray")
ax.set_xlim(-6, 6)
ax.set_title(r"(c) Ground-state $\psi_0$,  $A=0.5,\,w=2\rho_0$")
ax.grid(True, alpha=0.25)
ax.legend(handles=[line_psi, line_rho], loc="upper right",
          fontsize=9, framealpha=0.95)

plt.suptitle(
    r"Reduced 1D $\eta$-mode defect localization",
    fontsize=12, y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.97])

out_path = os.path.join(output_dir, "fig03_defect_localization.png")
plt.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

# Numerical summary table
print()
print("Numerical results (12 cases):")
print(f"  {'geometry':8s}  {'profile':18s}  {'E_min':>12s}  {'M_0':>12s}  {'E/M0':>8s}")
for glabel in geom_labels:
    for plabel in profile_labels:
        E, M0 = results[f"{glabel}|{plabel}"]
        print(f"  {glabel:8s}  {plabel:18s}  {E:12.4f}  {M0:12.4f}  {E/M0:8.4f}")


## Fig. 4 — §4.2 / B2: $Nil^3$ Heisenberg Landau spectrum and APS construction

**Source:** Heisenberg ladder algebra + Levi-Civita spinor CS integral
**Section:** §4.2, B2

**Construction sketch:**

1. In each PPA $p_2$ Fourier sector ($p_2 \in \mathbb{Z}+1/2$), the $Nil^3$ Dirac operator
   reduces to a Heisenberg ladder with spectrum
   $$
   \mu_n^{\uparrow} = 2n\,\omega\,|k_2| + k_2^2,\qquad
   \mu_n^{\downarrow} = 2(n+1)\,\omega\,|k_2| + k_2^2,\qquad
   \omega = |k_2|/r_0.
   $$
   The pairing $\mu_{n+1}^{\uparrow} = \mu_n^{\downarrow}$ leaves the
   $n=0$ spin-$\uparrow$ branch unpaired, contributing the spectral asymmetry.
2. Direct numerical evaluation of the Levi-Civita spinor CS integral on $Nil^3$ gives
   $\int_Y \mathcal{C}_3^{\sigma} = -\tfrac12$ (this notebook).
3. The DPPU sign convention $\eta_{\rm APS} = -\!\int_Y \mathcal{C}_3^{\sigma}$
   then yields $\eta_{\rm APS}^{(3D)}(Nil^3) = +\tfrac12$; the PPA spin
   structure has $h = \dim\ker D = 0$ (the $p_2 = 0$ zero mode is excluded),
   so $\eta(0) = 2\eta_{\rm APS} - h = 1$.

**Panel layout:** Heisenberg Landau levels for the lowest few PPA modes
$p_2 \in \{\pm\tfrac12,\pm\tfrac32,\pm\tfrac52\}$, with the unpaired
$n=0$ spin-$\uparrow$ branch highlighted. The APS construction chain is
kept in the notebook text and paper text rather than as a separate panel.


In [ ]:
# ── Fig. 4 — Nil^3 Heisenberg Landau spectrum and APS construction ──────
plt.close("all")

ppa_modes = fig4["ppa_modes"]
spectrum = fig4["spectrum"]
r0_nil = fig4["r0"]
n_max_show = fig4["n_max_show"]

fig, ax = plt.subplots(figsize=(8.4, 5.4))

# ── Heisenberg Landau spectrum ───────────────────────────────────────────

color_unpaired = "#F44336"
color_up_n     = "#FFB300"
color_down     = "#1976D2"

for i_p, p2 in enumerate(ppa_modes):
    s = spectrum[str(p2)]
    up = s["up"]; down = s["down"]
    for n in range(n_max_show + 1):
        if n == 0:
            ax.scatter([p2], [up[n]], color=color_unpaired, marker="o", s=110,
                       edgecolor="black", linewidths=0.8, zorder=5,
                       label=(r"$\mu_0^{\uparrow}$ (unpaired)"
                              if i_p == 0 else None))
        else:
            ax.scatter([p2], [up[n]], color=color_up_n, marker="s", s=55,
                       edgecolor="black", linewidths=0.5, zorder=4,
                       label=(r"$\mu_{n\geq1}^{\uparrow}$"
                              if i_p == 0 and n == 1 else None))
        ax.scatter([p2], [down[n]], color=color_down, marker="v", s=55,
                   edgecolor="black", linewidths=0.5, zorder=4,
                   label=(r"$\mu_n^{\downarrow}$"
                          if i_p == 0 and n == 0 else None))

# Pairing marker for one demo p_2 sector (positive, smallest)
demo_p2 = 0.5
s_demo = spectrum[str(demo_p2)]
ax.annotate(
    r"$\mu_{n+1}^{\uparrow} = \mu_n^{\downarrow}$",
    xy=(demo_p2, s_demo["down"][0]),
    xytext=(0.92, 7.2),
    arrowprops=dict(arrowstyle="->", color="#666666", lw=1.0),
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bbbbbb", alpha=0.9),
    fontsize=9.5,
    color="#555555",
)

ax.set_xlabel(r"PPA mode $p_2 \in \mathbb{Z} + 1/2$")
ax.set_ylabel(r"$\mu = \lambda^2$  ($r_0 = 3$)")
ax.set_title(r"Heisenberg Landau levels in each $p_2$ sector")
ax.legend(loc="upper center", ncol=3, fontsize=9, framealpha=0.95)
ax.grid(True, alpha=0.25)
ax.set_ylim(0, max(spectrum[str(2.5)]["up"][n_max_show], 18))

plt.suptitle(
    r"$Nil^3$ APS spectral core:  $\eta_{\rm APS}^{(3D)} = +1/2$",
    fontsize=12, y=1.02,
)
plt.tight_layout()
out_path = os.path.join(output_dir, "fig04_nil3_aps.png")
plt.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

print()
print(f"  int_Y CS_sigma  = {fig4['cs_sigma_total']:+.8f}  (expected -1/2)")
print(f"  eta_APS         = {fig4['eta_APS']:+.8f}  (expected +1/2)")
print(f"  h (PPA kernel)  = {fig4['h']}")
print(f"  eta(0)          = {fig4['eta0']:+.0f}")


## Fig. 5 — §4.2 / B4: $Sol^3$ global spectral branch

**Source:** local CS computation + hyperbolic monodromy + spin-structure kernel
**Section:** §4.2, B4

**Setup:** the compact Sol mapping torus
$M_A = T^2 \rtimes_A S^1$, $A = \begin{pmatrix}2&1\\1&1\end{pmatrix}$
with two spin structures `Sol-P` (base periodic) and `Sol-A` (base anti-periodic).

**Key observations:**

1. The local Levi-Civita spinor CS density vanishes:
   $\int_Y \mathcal{C}_3^{\sigma} = 0$ — so $Sol^3$ has *no* local APS core.
2. Time-reversal antiunitary symmetry $T D T^{-1} = -D$ (from real $E_a$ and
   $T = i\sigma_2 K$) gives $\eta(0)=0$ for both spin structures.
3. Nonzero fiber Fourier sectors $k\neq0$ have no zero modes:
   $m_k(t) = \|p_k(t)\| > 0$ for all $t$, so the reduced operator
   $(d/dt \pm m_k(t))$ admits no periodic / anti-periodic solutions.
4. The $k=0$ sector reduces to $D_0 = \gamma^3 d/dt$, whose constant-spinor
   zero modes descend only when the base circle is periodic:
   $h(\text{Sol-P}) = 2$, $h(\text{Sol-A}) = 0$.
5. Assembly: $\eta_{\rm APS} = (\eta(0) + h)/2$ gives
   $\eta_{\rm APS}(\text{Sol-P}) = 1$,
   $\eta_{\rm APS}(\text{Sol-A}) = 0$ —
   a global, spin-structure-dependent branch (not local).

**Panel layout:**

- (a) Fiber mass profile $m_k(t)$ for representative $k\neq0$ modes
  (log scale; all strictly positive).
- (b) Kernel dimension $h$ and assembled $\eta_{\rm APS}^{(3D)}$ for
  Sol-P vs Sol-A.


In [ ]:
# ── Fig. 5 — Sol^3 global spectral branch on the compact mapping torus ──
plt.close("all")

t_grid = np.array(fig5["t_grid"])
sample_modes = [tuple(k) for k in fig5["sample_modes"]]
m_profiles = {tuple(ast.literal_eval(s)): np.array(v)
              for s, v in fig5["m_profiles"].items()}
h_solp, h_sola = fig5["h_solp"], fig5["h_sola"]
eta_solp, eta_sola = fig5["eta_solp"], fig5["eta_sola"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

# ── (a) Fiber mass profile m_k(t) for nonzero k ──────────────────────────
ax = axes[0]
colors_k = ["#F44336", "#2196F3", "#4CAF50", "#9C27B0"]
for k, color in zip(sample_modes, colors_k):
    prof = m_profiles[k]
    ax.plot(t_grid, prof, lw=2.2, color=color,
            label=fr"$k=({k[0]},{k[1]})$")

ax.set_xlabel(r"$t \in [0,1]$  (base circle parameter)")
ax.set_ylabel(r"$m_k(t) = \|p_k(t)\|$")
ax.set_yscale("log")
ax.set_title(r"(a) Nonzero-$k$ fiber mass: $\,m_k(t) > 0$")
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, alpha=0.25, which="both")
ax.text(0.97, 0.05,
        r"$\Rightarrow$ no zero modes in $k\!\neq\!0$ sectors",
        transform=ax.transAxes, ha="right", fontsize=9.5,
        color="#666666", style="italic")

# ── (b) Kernel dimension h and eta_APS by spin structure ─────────────────
ax = axes[1]
spin_labels = ["Sol-P\n(base periodic)", "Sol-A\n(base anti-periodic)"]
xs = np.arange(2)
width = 0.35

hvals = [h_solp, h_sola]
evals = [eta_solp, eta_sola]

bars_h = ax.bar(xs - width/2, hvals, width,
                color="#1976D2", edgecolor="white",
                label=r"$h = \dim\ker D$")
bars_e = ax.bar(xs + width/2, evals, width,
                color="#F57C00", edgecolor="white",
                label=r"$\eta_{\rm APS}^{(3D)}$")
for bar in bars_h:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.07,
            f"{int(h)}", ha="center", va="bottom",
            fontsize=12, color="#1976D2", weight="bold")
for bar in bars_e:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.07,
            f"{h:.0f}", ha="center", va="bottom",
            fontsize=12, color="#F57C00", weight="bold")

ax.set_xticks(xs); ax.set_xticklabels(spin_labels, fontsize=10)
ax.set_ylabel(r"$h$  /  $\eta_{\rm APS}^{(3D)}$")
ax.set_ylim(0, 2.7)
ax.set_title(r"(b) Spin-structure dependent kernel and APS")
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.25, axis="y")

plt.suptitle(
    r"$Sol^3$ global spectral branch on $M_A = T^2 \rtimes_A S^1$,  "
    r"$A=[[2,1],[1,1]]$",
    fontsize=11.5, y=1.02,
)
plt.tight_layout()
out_path = os.path.join(output_dir, "fig05_sol3_global_spectral.png")
plt.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

print()
print(f"  int_Y CS_sigma (local)            = {fig5['cs_sigma_local']:+.4f}")
print(f"  Sol-P:  h = {h_solp},  eta_APS = {eta_solp:.0f}")
print(f"  Sol-A:  h = {h_sola},  eta_APS = {eta_sola:.0f}")
print(f"  m_k(t) range (over t and k):  "
      f"min = {min(min(p) for p in m_profiles.values()):.4f}, "
      f"max = {max(max(p) for p in m_profiles.values()):.4f}")


## Fig. 7 — §4.5: scaffold-vs-CS scatter

**Source:** §4.5 scaffold table
**Section:** §4.5

This figure is table-derived rather than engine-derived.  It plots the four
geometries with:

- horizontal axis: CS direction count $(0,1,3,0)$,
- vertical axis: normalized KK anisotropy $A_{\rm KK}/K^2$,
- marker size: Weyl scaffold datum $C^2_{\rm LC}$, recorded as
  $0,\,4/(3R^4),\,0,\,16/(3R^4)$,
- marker color: spin-2 rigidity tag.

The purpose is to make the $T^3$ / $Sol^3$ distinction visible: both have
CS direction count $0$, but $Sol^3$ sits on the nonzero anisotropy and
rigid-scaffold branch.


In [ ]:
# ── Fig. 7 — scaffold-vs-CS scatter ───────────────────────────────────
plt.close("all")

from matplotlib.lines import Line2D

points = [
    dict(key="T3",   label=r"$T^3$",   cs=0, akk=0.0,     c2=0.0,      tag="trivial"),
    dict(key="Nil3", label=r"$Nil^3$", cs=1, akk=2.0/3.0, c2=4.0/3.0,  tag="non-rigid"),
    dict(key="S3",   label=r"$S^3$",   cs=3, akk=0.0,     c2=0.0,      tag="non-rigid"),
    dict(key="Sol3", label=r"$Sol^3$", cs=0, akk=2.0/3.0, c2=16.0/3.0, tag="rigid"),
]

tag_color = {
    "trivial": "#94A3B8",
    "non-rigid": "#2563EB",
    "rigid": "#DC2626",
}

def marker_size(c2):
    return 180.0 + 130.0 * c2

fig, ax = plt.subplots(figsize=(7.8, 5.6))

for p in points:
    ax.scatter(
        p["cs"], p["akk"],
        s=marker_size(p["c2"]),
        color=tag_color[p["tag"]],
        edgecolor="white",
        linewidth=1.8,
        alpha=0.92,
        zorder=4,
    )

offsets = {
    "T3": (0.14, 0.045),
    "Nil3": (0.12, -0.055),
    "S3": (-0.42, 0.045),
    "Sol3": (0.14, -0.055),
}
c2_labels = {
    "T3":   r"$C^2_{\rm LC}=0$",
    "Nil3": r"$C^2_{\rm LC}=4/(3R^4)$",
    "S3":   r"$C^2_{\rm LC}=0$",
    "Sol3": r"$C^2_{\rm LC}=16/(3R^4)$",
}
for p in points:
    dx, dy = offsets[p["key"]]
    ax.text(
        p["cs"] + dx, p["akk"] + dy,
        p["label"] + "\n" + p["tag"] + "\n" + c2_labels[p["key"]],
        fontsize=10.5,
        ha="left" if dx >= 0 else "right",
        va="center",
        color="#111827",
        bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.72),
    )

ax.annotate(
    "",
    xy=(0, 2.0/3.0),
    xytext=(0, 0.025),
    arrowprops=dict(arrowstyle="<->", lw=1.1, color="#475569"),
)
ax.text(
    -0.28, 0.34,
    "same CS count\nscaffold separates",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#475569",
)

ax.set_xlim(-0.45, 3.45)
ax.set_ylim(-0.08, 0.78)
ax.set_xticks([0, 1, 3])
ax.set_xticklabels(["0", "1", "3"])
ax.set_yticks([0, 2.0/3.0])
ax.set_yticklabels(["0", r"$2/3$"])
ax.set_xlabel("CS direction count")
ax.set_ylabel(r"$A_{\rm KK}/K^2$")
ax.set_title(r"scaffold-vs-CS map for the four geometries")
ax.grid(True, alpha=0.25, zorder=1)

color_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=tag_color[tag],
           markeredgecolor="white", markeredgewidth=1.4, markersize=10, label=tag)
    for tag in ["trivial", "non-rigid", "rigid"]
]
leg1 = ax.legend(handles=color_handles, title="spin-2 rigidity tag",
                 loc="upper right", framealpha=0.95, fontsize=9)
ax.add_artist(leg1)

plt.tight_layout()
out_path = os.path.join(output_dir, "fig07_scaffold_vs_cs.png")
plt.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


## Summary

| Figure | File | Section / Rule | Source |
|--------|------|----------------|--------|
| Fig. 2 | `LaTeX/figures/fig02_ec_slice_potential.png` | §3.3 / R4 | `dppu.action.ec_action.build_veff_ec` |
| Fig. 3 | `LaTeX/figures/fig03_defect_localization.png` | §3.4 / R5 | `dppu` engine + SL eigensolver |
| Fig. 4 | `LaTeX/figures/fig04_nil3_aps.png` | §4.2 / B2 | Heisenberg ladder + Levi-Civita spinor CS |
| Fig. 5 | `LaTeX/figures/fig05_sol3_global_spectral.png` | §4.2 / B4 | local CS + hyperbolic monodromy + spin-structure kernel |
| Fig. 7 | `LaTeX/figures/fig07_scaffold_vs_cs.png` | §4.5 | scaffold table |

**Cache:** `data/paper04_figures_cache.pkl`
Delete the cache file to force a fresh engine run.

**Reproducibility:** the figure data is derived from the same DPPU library and standalone routines used in the paper04 verification scripts:

- `scripts/paper04/eta_defect_coefficients.py`
- `scripts/paper04/defect_localization.py`
- `scripts/paper04/eta_aps_nil3.py`
- `scripts/paper04/ec_slice_minima.py`
- `scripts/proofs/eta_aps_sol3.py`
- `scripts/proofs/landau_levels_nil3.py`
